# 🧠 Model 3 — Attention-Based Multimodal Drug Recognition
> **Pipeline**: Image (MobileNet/EfficientNet backbone) + Text (BERT/DistilBERT backbone) → Cross-Modal Attention Fusion → Drug Classification
>
> **Output**: Trained model artifacts ready for deployment (`.keras` + `label_encoder.npy`)
>
> **Author**: Auto-generated scaffold — fill in your paths and run cell by cell

---
## 📦 Cell 1 — Install & Import Dependencies

In [11]:
import sys
!{sys.executable} -m pip install numpy matplotlib tensorflow scikit-learn transformers==4.49.0 pillow seaborn


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import sys
!{sys.executable} -m pip install --upgrade keras tensorflow


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# ── Install (run once, comment out after) ──────────────────────────────────
# !pip install tensorflow tf_keras transformers scikit-learn numpy matplotlib pillow seaborn

import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer
from PIL import Image
import seaborn as sns

warnings.filterwarnings('ignore')
print(f'TensorFlow  : {tf.__version__}')
print(f'Keras       : {keras.__version__}')
print(f'GPU devices : {tf.config.list_physical_devices("GPU")}')

TensorFlow  : 2.21.0
Keras       : 3.14.1
GPU devices : []


---
## ⚙️ Cell 2 — Global Configuration
> **Edit this cell only** — all paths and hyper-parameters live here.

In [14]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Root folder (the folder you see in VS Code)
BASE_DIR = 'saved_models_final'

# Model 1 (EfficientNet + DistilBERT)
M1_FUSION_PATH   = os.path.join(BASE_DIR, 'model1', 'fusion_best.keras')
M1_TOKENIZER_DIR = os.path.join(BASE_DIR, 'model1', 'tokenizer')
M1_LABEL_ENCODER = os.path.join(BASE_DIR, 'model1', 'label_encoder.npy')

# Model 2 (MobileNet + BERT)
M2_FUSION_PATH   = os.path.join(BASE_DIR, 'model2', 'fusion_best.keras')
M2_TOKENIZER_DIR = os.path.join(BASE_DIR, 'model2', 'tokenizer')
M2_LABEL_ENCODER = os.path.join(BASE_DIR, 'model2', 'label_encoder.npy')

# Use Model 1's label encoder (both should be identical)
LABEL_ENCODER_PATH = M1_LABEL_ENCODER

# Dataset paths — matches your structure: dataset/processed/train|val|test/<class>/
DATASET_PROCESSED = os.path.join('dataset', 'processed')
DATASET_TRAIN = os.path.join(DATASET_PROCESSED, 'train')
DATASET_VAL   = os.path.join(DATASET_PROCESSED, 'val')
DATASET_TEST  = os.path.join(DATASET_PROCESSED, 'test')
# Text descriptions JSON
DESC_JSON = os.path.join('dataset', 'text', 'class_descriptions.json')

# Output directory for Model 3 artifacts (will be created next to saved_models_final)
OUTPUT_DIR = 'model3_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Sanity check — verify all critical files exist ─────────────────────────
critical_files = {
    'M1 fusion model'   : M1_FUSION_PATH,
    'M1 tokenizer dir'  : M1_TOKENIZER_DIR,
    'M1 label encoder'  : M1_LABEL_ENCODER,
    'M2 fusion model'   : M2_FUSION_PATH,
    'M2 tokenizer dir'  : M2_TOKENIZER_DIR,
    'M2 label encoder'  : M2_LABEL_ENCODER,
    'Train split'       : DATASET_TRAIN,
    'Val split'         : DATASET_VAL,
    'Test split'        : DATASET_TEST,
    'class_descriptions': DESC_JSON,
}
all_ok = True
for name, path in critical_files.items():
    exists = os.path.exists(path)
    flag   = '✅' if exists else '❌ NOT FOUND'
    print(f'  {flag}  {name:<22} → {path}')
    if not exists: all_ok = False
if not all_ok:
    raise FileNotFoundError('Fix the missing paths above before continuing.')

# ── Hyper-parameters ───────────────────────────────────────────────────────
IMG_SIZE        = (224, 224)      # both backbones expect 224×224
MAX_TEXT_LEN    = 128             # token length for both tokenizers
EMBED_DIM       = 256             # projection dim for attention space
NUM_HEADS       = 4               # multi-head attention heads
DROPOUT_RATE    = 0.3
BATCH_SIZE      = 16
EPOCHS          = 30
LEARNING_RATE   = 1e-4
PATIENCE        = 7               # early stopping patience

print('\n✅ Config loaded. Resolved paths:')
print(f'  M1 model  : {os.path.abspath(M1_FUSION_PATH)}')
print(f'  M2 model  : {os.path.abspath(M2_FUSION_PATH)}')
print(f'  Train     : {os.path.abspath(DATASET_TRAIN)}')
print(f'  Val       : {os.path.abspath(DATASET_VAL)}')
print(f'  Test      : {os.path.abspath(DATASET_TEST)}')
print(f'  Desc JSON : {os.path.abspath(DESC_JSON)}')
print(f'  Output    : {os.path.abspath(OUTPUT_DIR)}')

  ✅  M1 fusion model        → saved_models_final\model1\fusion_best.keras
  ✅  M1 tokenizer dir       → saved_models_final\model1\tokenizer
  ✅  M1 label encoder       → saved_models_final\model1\label_encoder.npy
  ✅  M2 fusion model        → saved_models_final\model2\fusion_best.keras
  ✅  M2 tokenizer dir       → saved_models_final\model2\tokenizer
  ✅  M2 label encoder       → saved_models_final\model2\label_encoder.npy
  ✅  Train split            → dataset\processed\train
  ✅  Val split              → dataset\processed\val
  ✅  Test split             → dataset\processed\test
  ✅  class_descriptions     → dataset\text\class_descriptions.json

✅ Config loaded. Resolved paths:
  M1 model  : c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\model1\fusion_best.keras
  M2 model  : c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\model2\fusion_best.keras
  Train     : c:\Users\hp\Documents\GitHub\deep_learning_project_final\dataset\pro

---
## 🔖 Cell 3 — Load Label Encoder & Discover Classes

In [15]:
# Load the label encoder saved from Model 1 / Model 2 training
le = LabelEncoder()
le.classes_ = np.load(LABEL_ENCODER_PATH, allow_pickle=True)
NUM_CLASSES = len(le.classes_)

print(f'Number of classes : {NUM_CLASSES}')
print(f'Classes           : {list(le.classes_)}')

Number of classes : 11
Classes           : [np.str_('analgesics_pain_fever'), np.str_('cardiovascular_blood'), np.str_('dermatology'), np.str_('diabetes_endocrine'), np.str_('eye_ear_nose_preparations'), np.str_('hormones_oncology'), np.str_('infections_immunity'), np.str_('neuro_musculo'), np.str_('respiratory_digestive'), np.str_('steroids_topicals'), np.str_('vitamins_supplements')]


---
## 🖼️ Cell 4 — Load Tokenizers (DistilBERT for M1, BERT for M2)

In [16]:
# Model 1 tokenizer  (DistilBERT)
tokenizer_m1 = AutoTokenizer.from_pretrained(M1_TOKENIZER_DIR)
print('✅ Model 1 tokenizer loaded:', tokenizer_m1.__class__.__name__)

# Model 2 tokenizer  (BERT)
tokenizer_m2 = AutoTokenizer.from_pretrained(M2_TOKENIZER_DIR)
print('✅ Model 2 tokenizer loaded:', tokenizer_m2.__class__.__name__)

✅ Model 1 tokenizer loaded: DistilBertTokenizerFast
✅ Model 2 tokenizer loaded: BertTokenizerFast


---
## 📂 Cell 5 — Dataset Loading
> **Assumption**: each class folder contains `.jpg`/`.png` images.  
> Text descriptions are loaded from a companion `descriptions.json` (one per image by filename) **or** auto-generated as `"drug image of <class_name>"` when missing.

In [17]:
# ── Load class_descriptions.json ──────────────────────────────────────────
# Format: { "class_name": { "description": "...", "note": "..." }, ... }
if os.path.exists(DESC_JSON):
    with open(DESC_JSON, encoding='utf-8') as f:
        raw_desc = json.load(f)
    # Build a flat dict: class_name → combined text string
    class_descriptions = {
        cls: f"{info.get('description','')} {info.get('note','')}".strip()
        for cls, info in raw_desc.items()
    }
    print(f'✅ Loaded descriptions for {len(class_descriptions)} classes:')
    for cls, txt in class_descriptions.items():
        print(f'  {cls:<30} → {txt[:60]}')
else:
    class_descriptions = {}
    print('⚠️  class_descriptions.json not found — using class-name fallback.')


# ── Helper: scan one split folder ─────────────────────────────────────────
SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def scan_split(split_dir):
    """Scan dataset/processed/{train|val|test}/<class>/*.jpg
    Returns (image_paths, text_labels, class_labels) lists."""
    imgs, txts, cls_list = [], [], []
    if not os.path.isdir(split_dir):
        raise NotADirectoryError(f'Split folder not found: {split_dir}')
    for class_name in sorted(os.listdir(split_dir)):
        class_dir = os.path.join(split_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        # Get text for this class from JSON, else build from class name
        text = class_descriptions.get(
            class_name,
            f'a pharmaceutical drug called {class_name.replace("_", " ")}'
        )
        for fname in os.listdir(class_dir):
            if os.path.splitext(fname)[1].lower() not in SUPPORTED_EXT:
                continue
            imgs.append(os.path.join(class_dir, fname))
            txts.append(text)          # same description for all images in class
            cls_list.append(class_name)
    return imgs, txts, cls_list


# ── Scan all three splits ──────────────────────────────────────────────────
train_img_raw, train_txt_raw, train_cls = scan_split(DATASET_TRAIN)
val_img_raw,   val_txt_raw,   val_cls   = scan_split(DATASET_VAL)
test_img_raw,  test_txt_raw,  test_cls  = scan_split(DATASET_TEST)

print(f'\nSplit sizes  →  Train: {len(train_img_raw)}  |  Val: {len(val_img_raw)}  |  Test: {len(test_img_raw)}')
print(f'\nClass distribution (train):')
for cls, cnt in zip(*np.unique(train_cls, return_counts=True)):
    print(f'  {cls:<35} {cnt}')

✅ Loaded descriptions for 11 classes:
  dermatology                    → Skin conditions — acne, eczema, etc. Follow application inst
  neuro_musculo                  → Neurology, psychiatry, musculoskeletal. Do not stop abruptly
  hormones_oncology              → Hormones, oncology, anaesthetics. Strict medical supervision
  diabetes_endocrine             → Blood sugar & endocrine disorders. Monitor glucose regularly
  cardiovascular_blood           → Heart & blood-pressure medicines. Follow dosage strictly.
  analgesics_pain_fever          → Pain relief and fever reduction. Do not exceed recommended d
  respiratory_digestive          → Respiratory & digestive treatments. Take digestive meds arou
  infections_immunity            → Antibiotics, antifungals, immunology. Complete the full cour
  steroids_topicals              → Topical anti-inflammatory steroids. Avoid prolonged use.
  eye_ear_nose_preparations      → Eyes, ears, nose treatments. Avoid contamination of applicat
  vitamin

---
## ✂️ Cell 6 — Train / Validation / Test Split

In [18]:
# Your dataset is already split into train/val/test folders — no random split needed.
# We just encode the class labels using the LabelEncoder loaded in Cell 3.

train_img, train_txt = train_img_raw, train_txt_raw
val_img,   val_txt   = val_img_raw,   val_txt_raw
test_img,  test_txt  = test_img_raw,  test_txt_raw

train_y = le.transform(train_cls)
val_y   = le.transform(val_cls)
test_y  = le.transform(test_cls)

print(f'Train : {len(train_img)}  |  Val : {len(val_img)}  |  Test : {len(test_img)}')
print(f'Classes seen in train: {sorted(set(train_cls))}')

Train : 1407  |  Val : 401  |  Test : 214
Classes seen in train: ['analgesics_pain_fever', 'cardiovascular_blood', 'dermatology', 'diabetes_endocrine', 'eye_ear_nose_preparations', 'hormones_oncology', 'infections_immunity', 'neuro_musculo', 'respiratory_digestive', 'steroids_topicals', 'vitamins_supplements']


---
## 🔄 Cell 7 — Data Generator
> Produces four inputs per sample:  
> `[image, m1_input_ids, m1_attention_mask, m2_input_ids, m2_attention_mask]`

In [19]:
def preprocess_image(path):
    """Load and normalise a single image to [0,1]."""
    img = Image.open(path).convert('RGB').resize(IMG_SIZE)
    return np.array(img, dtype=np.float32) / 255.0


def tokenize_batch(texts, tokenizer):
    """Return (input_ids, attention_mask) as float32 numpy arrays."""
    enc = tokenizer(
        texts,
        max_length=MAX_TEXT_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='np'
    )
    return enc['input_ids'].astype(np.float32), enc['attention_mask'].astype(np.float32)


def build_tf_dataset(img_paths, texts, labels, shuffle=False):
    """
    Returns a tf.data.Dataset that yields:
      inputs = (image, m1_ids, m1_mask, m2_ids, m2_mask)
      target = one-hot label
    """
    n = len(img_paths)
    # Pre-tokenise entire split (fits in RAM for typical drug datasets)
    m1_ids,  m1_mask  = tokenize_batch(texts, tokenizer_m1)
    m2_ids,  m2_mask  = tokenize_batch(texts, tokenizer_m2)
    images = np.stack([preprocess_image(p) for p in img_paths])   # (N,224,224,3)
    one_hot = tf.keras.utils.to_categorical(labels, NUM_CLASSES)

    ds = tf.data.Dataset.from_tensor_slices(
        ((images, m1_ids, m1_mask, m2_ids, m2_mask), one_hot)
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=n, seed=42)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


print('Building datasets (this may take a minute)…')
train_ds = build_tf_dataset(train_img, train_txt, train_y, shuffle=True)
val_ds   = build_tf_dataset(val_img,   val_txt,   val_y)
test_ds  = build_tf_dataset(test_img,  test_txt,  test_y)
print('✅ Datasets ready.')

Building datasets (this may take a minute)…
✅ Datasets ready.


---
## 🏗️ Cell 8 — Load Pretrained Fusion Models (Embedding Extractors)
> We strip the final softmax layer and expose the **embedding layer** from each model.

In [20]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8 — Load Pretrained Fusion Models (Fixed)
# ═══════════════════════════════════════════════════════════════════════════
#
# ROOT CAUSE OF THE ERROR
# ───────────────────────
# Strategy 1 fails for M2 because fusion_model.keras was saved with tf_keras
# (module path 'tf_keras.src.engine.functional'), which tf.keras cannot
# deserialize in a different environment.
#
# Strategy 2 (load_weights) fails because it matches weights BY LAYER NAME.
# The skeleton uses generic names (batch_normalization_2, dropout_4…) but the
# saved M2 file has different auto-incremented names (batch_normalization_8,
# dropout_88…).  Zero variables are found → loading silently fails.
#
# THE FIX — Strategy 2b: weight copying BY POSITION (index order)
# ─────────────────────────────────────────────────────────────────
# We open the .keras archive (it is a ZIP), parse its config JSON to discover
# the real layer names and order, extract the weights.h5 from the same ZIP,
# and copy weights layer-by-layer IN INDEX ORDER into a freshly built skeleton.
# Position is stable regardless of the auto-incremented name suffix.

import os, json, zipfile, io, warnings
import numpy as np
import h5py
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

warnings.filterwarnings('ignore')


# ── Architecture builder ────────────────────────────────────────────────────
def _build_fusion_skeleton(num_classes=11, name='fusion'):
    """Recreate the exact fusion architecture (generic layer names are fine
    here because we copy weights BY POSITION, not by name)."""
    img_in = keras.Input(shape=(256,), name='image_features_in')
    txt_in = keras.Input(shape=(256,), name='text_features_in')

    x = layers.Concatenate(name='fusion_concat')([img_in, txt_in])          # 512

    x = layers.Dense(256, use_bias=True, name='mlp_256')(x)
    x = layers.BatchNormalization(momentum=0.99, epsilon=0.001,
                                  name='bn_256')(x)
    x = layers.Activation('relu', name='relu_256')(x)
    x = layers.Dropout(0.5, name='drop_256')(x)

    x = layers.Dense(128, use_bias=True, name='mlp_128')(x)
    x = layers.BatchNormalization(momentum=0.99, epsilon=0.001,
                                  name='bn_128')(x)
    x = layers.Activation('relu', name='relu_128')(x)
    x = layers.Dropout(0.3, name='drop_128')(x)                              # ← embedding

    out = layers.Dense(num_classes, activation='softmax',
                       name='fusion_output')(x)

    return Model(inputs=[img_in, txt_in], outputs=out, name=name)


# ── Strategy 2b: positional weight copy from .keras ZIP ────────────────────
def _load_weights_by_position(skeleton, keras_path):
    """
    Open the .keras ZIP, read weights.h5, and copy weights into `skeleton`
    layer-by-layer IN INDEX ORDER (ignoring layer names entirely).
    Only layers that actually have weights are considered.
    """
    with zipfile.ZipFile(keras_path, 'r') as zf:
        names = zf.namelist()
        wname = next((n for n in names if n.endswith('.weights.h5')
                      or n == 'model.weights.h5' or 'weights' in n), None)
        if wname is None:
            raise FileNotFoundError(
                f'No weights file found inside {keras_path}. '
                f'Contents: {names}')
        raw = zf.read(wname)

    # Parse the weights HDF5 from memory
    with h5py.File(io.BytesIO(raw), 'r') as f:
        # Collect all weight arrays in file order
        saved_weights = []
        def _collect(name, obj):
            if isinstance(obj, h5py.Dataset):
                saved_weights.append((name, obj[()]))
        f.visititems(_collect)

    # Collect layers with weights from the skeleton (in model order)
    skeleton_layers_with_w = [l for l in skeleton.layers if l.weights]

    # Build a flat list of arrays from the HDF5 in the order they appear
    # (h5py visits in creation order which mirrors layer order)
    saved_arrays = [arr for _, arr in saved_weights]

    # Assign sequentially
    idx = 0
    for layer in skeleton_layers_with_w:
        n = len(layer.weights)
        chunk = saved_arrays[idx: idx + n]
        if len(chunk) != n:
            raise ValueError(
                f'Layer {layer.name} expects {n} weight arrays but only '
                f'{len(saved_arrays) - idx} remain in the saved file.')
        layer.set_weights(chunk)
        idx += n

    if idx != len(saved_arrays):
        print(f'  ℹ️  {len(saved_arrays) - idx} extra arrays in saved file '
              f'(possibly optimizer state) — ignored.')


# ── Main loader ─────────────────────────────────────────────────────────────
def load_fusion_model(path, num_classes=11, model_label='model'):
    """
    Try strategies in order:
      1. tf.keras.models.load_model        — works when envs match
      2. Rebuild skeleton + load_weights   — works when layer names match
      2b. Rebuild skeleton + positional weight copy from .keras ZIP
              — works even when layer names differ (the M2 case)
      3. Rebuild skeleton + load from .weights.h5 sidecar
    """
    print(f'\nLoading {model_label} from: {path}')

    # ── Strategy 1 ──────────────────────────────────────────────────────────
    try:
        model = tf.keras.models.load_model(path, compile=False)
        print(f'  ✅ Strategy 1 succeeded (tf.keras.models.load_model)')
        return model
    except Exception as e1:
        print(f'  ⚠️  Strategy 1 failed: {type(e1).__name__}: {str(e1)[:120]}')

    # ── Strategy 2: standard load_weights (name-based) ───────────────────
    try:
        skeleton = _build_fusion_skeleton(num_classes=num_classes, name=model_label)
        skeleton.load_weights(path)
        print(f'  ✅ Strategy 2 succeeded (rebuild + load_weights)')
        return skeleton
    except Exception as e2:
        print(f'  ⚠️  Strategy 2 failed: {type(e2).__name__}: {str(e2)[:120]}')

    # ── Strategy 2b: positional weight copy from ZIP ─────────────────────
    try:
        skeleton = _build_fusion_skeleton(num_classes=num_classes, name=model_label)
        _load_weights_by_position(skeleton, path)
        print(f'  ✅ Strategy 2b succeeded (rebuild + positional weight copy)')
        return skeleton
    except Exception as e2b:
        print(f'  ⚠️  Strategy 2b failed: {type(e2b).__name__}: {str(e2b)[:200]}')

    # ── Strategy 3: .weights.h5 sidecar ──────────────────────────────────
    h5_path = path.replace('.keras', '.weights.h5')
    if os.path.exists(h5_path):
        try:
            skeleton = _build_fusion_skeleton(num_classes=num_classes, name=model_label)
            skeleton.load_weights(h5_path)
            print(f'  ✅ Strategy 3 succeeded (rebuild + {h5_path})')
            return skeleton
        except Exception as e3:
            print(f'  ⚠️  Strategy 3 failed: {type(e3).__name__}: {str(e3)[:120]}')

    raise RuntimeError(
        f'All loading strategies failed for {path}.\n'
        'Re-save the model with: model.save("path.keras") '
        'using the SAME Python / TensorFlow environment that trained it, '
        'then copy here and retry.'
    )


# ── Load both fusion models ──────────────────────────────────────────────────
NUM_CLASSES = len(le.classes_)   # defined in Cell 3

m1_tf = load_fusion_model(M1_FUSION_PATH, num_classes=NUM_CLASSES, model_label='m1_fusion')
m2_tf = load_fusion_model(M2_FUSION_PATH, num_classes=NUM_CLASSES, model_label='m2_fusion')

print('\n✅ Both fusion models loaded. Layer summary:')
print('\n── M1 ──')
for layer in m1_tf.layers:
    print(f'  {layer.name:<40} {layer.output_shape}')

print('\n── M2 ──')
for layer in m2_tf.layers:
    print(f'  {layer.name:<40} {layer.output_shape}')



Loading m1_fusion from: saved_models_final\model1\fusion_best.keras
  ⚠️  Strategy 1 failed: TypeError: Could not deserialize class 'Functional' because its parent module keras.src.engine.functional cannot be imported. Full 
  ⚠️  Strategy 2 failed: ValueError: A total of 5 objects could not be loaded. Example error message for object <Dense name=mlp_256, built=True>:

Layer 'mlp
  ⚠️  Strategy 2b failed: ValueError: Layer mlp_256 weight shape (512, 256) is not compatible with provided weight shape (256,).
  ✅ Strategy 3 succeeded (rebuild + saved_models_final\model1\fusion_best.weights.h5)

Loading m2_fusion from: saved_models_final\model2\fusion_best.keras
  ⚠️  Strategy 1 failed: TypeError: Could not locate class 'Functional'. Make sure custom classes and functions are decorated with `@keras.saving.register_k
  ⚠️  Strategy 2 failed: ValueError: A total of 5 objects could not be loaded. Example error message for object <Dense name=mlp_256, built=True>:

Layer 'mlp
  ⚠️  Strategy 2b

RuntimeError: All loading strategies failed for saved_models_final\model2\fusion_best.keras.
Re-save the model with: model.save("path.keras") using the SAME Python / TensorFlow environment that trained it, then copy here and retry.

---
## 🔍 Cell 9 — Extract Embedding Sub-Models
> **Edit `EMB_LAYER_M1` / `EMB_LAYER_M2`** to match the second-to-last Dense layer name printed above.

In [ ]:
# ── Known architecture (read from error traceback) ────────────────────────
# Both M1 and M2 fusion models have identical structure:
#   Inputs: image_features_in(256) + text_features_in(256)
#   Dense(256, linear) → BN → ReLU → Dropout(0.5)
#   Dense(128, linear) → BN → ReLU → Dropout(0.3)   ← EMBEDDING OUTPUT (128-dim)
#   Dense(11, softmax)                                ← classification head (stripped)
#
# We rebuild both as keras 3 models taking (img_emb_256, txt_emb_256)
# and outputting the 128-dim embedding BEFORE the final Dense(11).

# ── Embedding layer name (second-to-last layer before softmax) ─────────────
# Auto-detect: layer[-2] is the Dropout before fusion_output
M1_EMB_LAYER = m1_tf.layers[-2].name   # e.g. 'dropout_43' or similar
M2_EMB_LAYER = m2_tf.layers[-2].name   # e.g. 'dropout_44'
print(f'M1 embedding layer : {M1_EMB_LAYER}  shape={m1_tf.layers[-2].output_shape}')
print(f'M2 embedding layer : {M2_EMB_LAYER}  shape={m2_tf.layers[-2].output_shape}')

EMB_DIM_FUSION = m1_tf.layers[-2].output_shape[-1]   # 128

# ── Rebuild embedding sub-models in keras 3 ───────────────────────────────
def rebuild_fusion_embedder(tf_model, emb_layer_name, name):
    """
    Recreates the fusion MLP in keras 3 and copies weights from tf_keras model.
    Returns a keras 3 Model: (img_feat_256, txt_feat_256) → embedding_128
    """
    # Inputs
    img_in = layers.Input(shape=(256,), name='image_features_in')
    txt_in = layers.Input(shape=(256,), name='text_features_in')

    # Rebuild layers in same order as original
    x = layers.Concatenate(name='fusion_concat')([img_in, txt_in])   # (512)

    x = layers.Dense(256, activation='linear',   name='mlp_256')(x)
    x = layers.BatchNormalization(momentum=0.99, epsilon=0.001, name='bn_256')(x)
    x = layers.Activation('relu',                name='relu_256')(x)
    x = layers.Dropout(0.5,                      name='drop_256')(x)

    x = layers.Dense(128, activation='linear',   name='mlp_128')(x)
    x = layers.BatchNormalization(momentum=0.99, epsilon=0.001, name='bn_128')(x)
    x = layers.Activation('relu',                name='relu_128')(x)
    x = layers.Dropout(0.3,                      name='drop_128')(x)   # ← embedding output

    model = Model(inputs=[img_in, txt_in], outputs=x, name=name)

    # ── Copy weights layer by layer ────────────────────────────────────────
    # Map: tf_keras layer name → keras 3 layer name
    weight_map = {}
    for tf_layer in tf_model.layers:
        w = tf_layer.get_weights()
        if not w: continue
        weight_map[tf_layer.name] = w

    name_remap = {
        'mlp_256': 'mlp_256',
        'mlp_128': 'mlp_128',
    }
    # BatchNorm layers — find by index
    bn_layers_tf = [l for l in tf_model.layers if 'batch_normalization' in l.name]
    bn_layers_k3 = [l for l in model.layers      if l.name.startswith('bn_')]

    for tf_l, k3_l in zip(bn_layers_tf, bn_layers_k3):
        k3_l.set_weights(tf_l.get_weights())
        print(f'  ✅ Copied BN  {tf_l.name} → {k3_l.name}')

    for tf_name, k3_name in name_remap.items():
        if tf_name in weight_map:
            model.get_layer(k3_name).set_weights(weight_map[tf_name])
            print(f'  ✅ Copied Dense {tf_name} → {k3_name}')

    model.trainable = False
    return model


print('\nRebuilding M1 embedder...')
m1_emb_model = rebuild_fusion_embedder(m1_tf, M1_EMB_LAYER, 'model1_embedder')
print('\nRebuilding M2 embedder...')
m2_emb_model = rebuild_fusion_embedder(m2_tf, M2_EMB_LAYER, 'model2_embedder')

print(f'\n✅ M1 embedder output shape : {m1_emb_model.output.shape}')
print(f'✅ M2 embedder output shape : {m2_emb_model.output.shape}')

---
## 🧩 Cell 10 — Model 3 Architecture: Full Pipeline

```
TRUE PIPELINE (based on your saved model configs)

Raw Inputs
 ├─ image        (224×224×3)
 ├─ m1_input_ids / m1_attention_mask   (MAX_TEXT_LEN)
 └─ m2_input_ids / m2_attention_mask   (MAX_TEXT_LEN)

Shared Image Feature Extractor  (EfficientNetB0, frozen)
 image → GlobalAvgPool → Dense(256) → img_feat   (256)

Text Feature Extractors  (mean-pool over token embeddings, frozen)
 m1_ids/mask → DistilBERT-style mean-pool → Dense(256) → txt_feat_m1  (256)
 m2_ids/mask → BERT-style mean-pool       → Dense(256) → txt_feat_m2  (256)

Frozen Fusion Embedders  (weights copied from M1/M2 fusion_best.keras)
 [img_feat, txt_feat_m1] → M1_FusionEmbedder → emb1  (128)
 [img_feat, txt_feat_m2] → M2_FusionEmbedder → emb2  (128)

Cross-Modal Attention
 proj1 = Dense(EMBED_DIM)(emb1)   proj2 = Dense(EMBED_DIM)(emb2)
 attended1 = MHA(q=proj1, k=proj2, v=proj2)   — M1 attends to M2
 attended2 = MHA(q=proj2, k=proj1, v=proj1)   — M2 attends to M1
 attended1 += proj1  (residual)   attended2 += proj2

Adaptive Gate  (sample-wise α/β)
 gate = Softmax(Dense(2)(concat([attended1, attended2])))
 fused = α·attended1 + β·attended2

Classifier Head
 fused → LayerNorm → Dense(256,relu) → Dropout → Dense(128,relu) → Dense(11,softmax)
```

In [ ]:
# ── Shared image feature extractor (EfficientNetB0) ──────────────────────
eff_base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights='imagenet',
    input_shape=(*IMG_SIZE, 3), pooling='avg'
)
eff_base.trainable = False
print(f'EfficientNetB0 output : {eff_base.output.shape}')   # (None, 1280)


def build_model3(num_classes, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                 dropout=DROPOUT_RATE):
    """
    Full Model 3 pipeline:
      Raw image + tokenised text (×2) → feature extraction
      → frozen M1/M2 fusion embedders → cross-modal attention
      → adaptive gate → classifier head → softmax

    Inputs (5 tensors):
      image          : (B, 224, 224, 3)
      m1_input_ids   : (B, MAX_TEXT_LEN)  int/float  DistilBERT token ids
      m1_attn_mask   : (B, MAX_TEXT_LEN)
      m2_input_ids   : (B, MAX_TEXT_LEN)  BERT token ids
      m2_attn_mask   : (B, MAX_TEXT_LEN)
    Output:
      (B, num_classes)  softmax probabilities
    """

    # ── 1. Raw inputs ──────────────────────────────────────────────────────
    img_input = layers.Input(shape=(*IMG_SIZE, 3),  name='image')
    m1_ids    = layers.Input(shape=(MAX_TEXT_LEN,), name='m1_input_ids')
    m1_mask   = layers.Input(shape=(MAX_TEXT_LEN,), name='m1_attention_mask')
    m2_ids    = layers.Input(shape=(MAX_TEXT_LEN,), name='m2_input_ids')
    m2_mask   = layers.Input(shape=(MAX_TEXT_LEN,), name='m2_attention_mask')

    # ── 2. Shared image feature: EfficientNetB0 → Dense(256) ──────────────
    img_feat_raw = eff_base(img_input, training=False)               # (B, 1280)
    img_feat     = layers.Dense(256, activation='relu',
                                name='img_proj')(img_feat_raw)        # (B, 256)

    # ── 3. Text features: mean-pool token embeddings → Dense(256) ─────────
    # We use a trainable Embedding + mean-pool as a lightweight substitute
    # for the frozen transformer; the fusion embedders were trained on 256-dim
    # text features so we must match that dimension.
    # Vocab sizes: DistilBERT ≈ 30522, BERT ≈ 30522
    VOCAB_SIZE = 30522
    TOKEN_EMB_DIM = 128   # internal token embedding dim

    # M1 text branch  (DistilBERT tokeniser)
    m1_tok_emb   = layers.Embedding(VOCAB_SIZE, TOKEN_EMB_DIM,
                                     mask_zero=True, name='m1_token_emb')(m1_ids)
    m1_mask_exp  = layers.Lambda(
        lambda m: tf.cast(tf.expand_dims(m, -1), tf.float32),
        name='m1_mask_expand')(m1_mask)
    m1_masked    = layers.Multiply(name='m1_masked')([m1_tok_emb, m1_mask_exp])
    m1_summed    = layers.Lambda(
        lambda x: tf.reduce_sum(x, axis=1), name='m1_sum')(m1_masked)
    m1_count     = layers.Lambda(
        lambda m: tf.maximum(tf.reduce_sum(m, axis=1), 1.0),
        name='m1_count')(m1_mask_exp)
    m1_mean      = layers.Lambda(
        lambda x: x[0] / x[1], name='m1_mean')([m1_summed, m1_count])
    txt_feat_m1  = layers.Dense(256, activation='relu',
                                 name='m1_txt_proj')(m1_mean)         # (B, 256)

    # M2 text branch  (BERT tokeniser)
    m2_tok_emb   = layers.Embedding(VOCAB_SIZE, TOKEN_EMB_DIM,
                                     mask_zero=True, name='m2_token_emb')(m2_ids)
    m2_mask_exp  = layers.Lambda(
        lambda m: tf.cast(tf.expand_dims(m, -1), tf.float32),
        name='m2_mask_expand')(m2_mask)
    m2_masked    = layers.Multiply(name='m2_masked')([m2_tok_emb, m2_mask_exp])
    m2_summed    = layers.Lambda(
        lambda x: tf.reduce_sum(x, axis=1), name='m2_sum')(m2_masked)
    m2_count     = layers.Lambda(
        lambda m: tf.maximum(tf.reduce_sum(m, axis=1), 1.0),
        name='m2_count')(m2_mask_exp)
    m2_mean      = layers.Lambda(
        lambda x: x[0] / x[1], name='m2_mean')([m2_summed, m2_count])
    txt_feat_m2  = layers.Dense(256, activation='relu',
                                 name='m2_txt_proj')(m2_mean)         # (B, 256)

    # ── 4. Frozen fusion embedders (pretrained MLP weights) ────────────────
    emb1 = m1_emb_model([img_feat, txt_feat_m1])   # (B, 128)
    emb2 = m2_emb_model([img_feat, txt_feat_m2])   # (B, 128)

    # ── 5. Project to common attention space ──────────────────────────────
    proj1 = layers.Dense(embed_dim, activation='relu', name='proj_m1')(emb1)  # (B, E)
    proj2 = layers.Dense(embed_dim, activation='relu', name='proj_m2')(emb2)  # (B, E)

    # Reshape to (B, 1, E) for MultiHeadAttention
    proj1_seq = layers.Reshape((1, embed_dim), name='reshape_m1')(proj1)
    proj2_seq = layers.Reshape((1, embed_dim), name='reshape_m2')(proj2)

    # ── 6. Cross-modal attention ───────────────────────────────────────────
    att1_seq = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim // num_heads,
        dropout=dropout, name='cross_attn_m1_to_m2'
    )(query=proj1_seq, key=proj2_seq, value=proj2_seq)   # (B,1,E)

    att2_seq = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim // num_heads,
        dropout=dropout, name='cross_attn_m2_to_m1'
    )(query=proj2_seq, key=proj1_seq, value=proj1_seq)   # (B,1,E)

    att1 = layers.Reshape((embed_dim,), name='squeeze_att1')(att1_seq)   # (B,E)
    att2 = layers.Reshape((embed_dim,), name='squeeze_att2')(att2_seq)   # (B,E)

    # Residual + LayerNorm
    att1 = layers.LayerNormalization(name='ln_m1')(layers.Add(name='res_m1')([proj1, att1]))
    att2 = layers.LayerNormalization(name='ln_m2')(layers.Add(name='res_m2')([proj2, att2]))

    # ── 7. Adaptive gate ──────────────────────────────────────────────────
    gate   = layers.Softmax(name='gate_softmax')(
                 layers.Dense(2, name='gate_logits')(
                     layers.Concatenate(name='gate_concat')([att1, att2])))
    alpha  = layers.Lambda(lambda g: g[:, 0:1], name='alpha')(gate)
    beta   = layers.Lambda(lambda g: g[:, 1:2], name='beta')(gate)
    fused  = layers.Add(name='fused')([
        layers.Multiply(name='w_m1')([att1, alpha]),
        layers.Multiply(name='w_m2')([att2, beta])
    ])

    # ── 8. Classifier head ────────────────────────────────────────────────
    x = layers.LayerNormalization(name='ln_fused')(fused)
    x = layers.Dense(256, activation='relu',  name='cls_d1')(x)
    x = layers.Dropout(dropout,               name='cls_drop1')(x)
    x = layers.Dense(128, activation='relu',  name='cls_d2')(x)
    x = layers.Dropout(dropout / 2,           name='cls_drop2')(x)
    out = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return Model(
        inputs  = [img_input, m1_ids, m1_mask, m2_ids, m2_mask],
        outputs = out,
        name    = 'Model3_AttentionFusion'
    )


model3 = build_model3(NUM_CLASSES)
model3.summary(expand_nested=False)
trainable   = sum(np.prod(v.shape) for v in model3.trainable_variables)
untrainable = sum(np.prod(v.shape) for v in model3.non_trainable_variables)
print(f'\nTrainable params   : {trainable:,}')
print(f'Frozen params      : {untrainable:,}')

---
## ⚡ Cell 11 — Compile Model 3

In [ ]:
optimizer = keras.optimizers.AdamW(
    learning_rate=LEARNING_RATE,
    weight_decay=1e-4
)

model3.compile(
    optimizer = optimizer,
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy',
                 keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

print('✅ Model compiled.')

---
## 🏋️ Cell 12 — Training with Callbacks

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath  = os.path.join(OUTPUT_DIR, 'model3_best.keras'),
        monitor   = 'val_accuracy',
        save_best_only= True,
        verbose   = 1
    ),
    keras.callbacks.EarlyStopping(
        monitor   = 'val_accuracy',
        patience  = PATIENCE,
        restore_best_weights=True,
        verbose   = 1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor   = 'val_loss',
        factor    = 0.5,
        patience  = 3,
        min_lr    = 1e-7,
        verbose   = 1
    ),
    keras.callbacks.CSVLogger(
        os.path.join(OUTPUT_DIR, 'model3_training_log.csv')
    )
]

history = model3.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = EPOCHS,
    callbacks       = callbacks,
    verbose         = 1
)

print('\n✅ Training complete.')

---
## 📈 Cell 13 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train Acc',  linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Acc',    linewidth=2)
axes[0].set_title('Model 3 — Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss',   linewidth=2)
axes[1].set_title('Model 3 — Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model3_training_history.png'), dpi=150)
plt.show()
print('✅ Plot saved.')

---
## 🧪 Cell 14 — Evaluate on Test Set

In [ ]:
# Load best checkpoint
best_model = keras.models.load_model(
    os.path.join(OUTPUT_DIR, 'model3_best.keras'), compile=False
)
best_model.compile(optimizer='adam', loss='categorical_crossentropy',
                   metrics=['accuracy'])

test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
print(f'\n Test Loss     : {test_loss:.4f}')
print(f' Test Accuracy : {test_acc*100:.2f}%')

# Per-class report
y_pred_proba = best_model.predict(test_ds, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)

print('\nClassification Report:')
report = classification_report(test_y, y_pred, target_names=le.classes_)
print(report)

with open(os.path.join(OUTPUT_DIR, 'model3_classification_report.txt'), 'w') as f:
    f.write(report)

---
## 🗺️ Cell 15 — Confusion Matrix

In [ ]:
cm = confusion_matrix(test_y, y_pred)
fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES), max(6, NUM_CLASSES - 2)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=ax, linewidths=0.5)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True',      fontsize=12)
ax.set_title('Model 3 — Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model3_confusion_matrix.png'), dpi=150)
plt.show()

---
## 🔬 Cell 16 — Attention Gate Analysis (per sample α / β)
> Visualise how much each model contributes for individual test samples.

In [ ]:
# Build a sub-model that outputs [predictions, gate_weights]
gate_model = Model(
    inputs  = best_model.inputs,
    outputs = [best_model.output,
               best_model.get_layer('gate_softmax').output],
    name    = 'gate_inspector'
)

# Run on the first test batch
first_batch  = next(iter(test_ds))
batch_inputs = first_batch[0]  # tuple of 5 tensors
batch_labels = first_batch[1]

preds, gates = gate_model.predict(batch_inputs, verbose=0)
alphas = gates[:, 0]   # M1 weight
betas  = gates[:, 1]   # M2 weight

print(f'Mean α (Model1 / text-strong) : {alphas.mean():.3f}')
print(f'Mean β (Model2 / image-strong): {betas.mean():.3f}')

# Bar chart of gate weights for first 16 samples
n_show = min(16, len(alphas))
x = np.arange(n_show)
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(x - 0.2, alphas[:n_show], 0.4, label='α  M1 (text-strong)',  color='steelblue')
ax.bar(x + 0.2, betas[:n_show],  0.4, label='β  M2 (image-strong)', color='coral')
ax.set_xticks(x); ax.set_xticklabels([f'S{i}' for i in range(n_show)])
ax.set_ylabel('Gate Weight'); ax.set_ylim(0, 1)
ax.set_title('Attention Gate Weights per Sample', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model3_gate_analysis.png'), dpi=150)
plt.show()

---
## 💾 Cell 17 — Save All Deployment Artifacts

In [ ]:
# 1. Best Model 3 (already saved by checkpoint callback)
best_model_path = os.path.join(OUTPUT_DIR, 'model3_best.keras')
print(f'✅ Best model  : {best_model_path}')

# 2. Label encoder
le_path = os.path.join(OUTPUT_DIR, 'label_encoder.npy')
np.save(le_path, le.classes_)
print(f'✅ Label encoder : {le_path}')

# 3. Model config JSON  (for deployment reference)
config = {
    'img_size'       : IMG_SIZE,
    'max_text_len'   : MAX_TEXT_LEN,
    'embed_dim'      : EMBED_DIM,
    'num_heads'      : NUM_HEADS,
    'num_classes'    : NUM_CLASSES,
    'classes'        : list(le.classes_),
    'm1_tokenizer'   : os.path.abspath(M1_TOKENIZER_DIR),
    'm2_tokenizer'   : os.path.abspath(M2_TOKENIZER_DIR),
    'emb_layer_m1'   : EMB_LAYER_M1,
    'emb_layer_m2'   : EMB_LAYER_M2,
}
config_path = os.path.join(OUTPUT_DIR, 'model3_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'✅ Config JSON   : {config_path}')

print('\n📦 Deployment artifacts ready in:', OUTPUT_DIR)

---
## 🔮 Cell 18 — Single-Sample Inference (Deployment-Ready Predictor)
> This function is the **exact code** your deployment app will call.

In [ ]:
class DrugPredictor:
    """
    Deployment-ready predictor for Model 3.
    
    Usage:
        predictor = DrugPredictor('model3_output/')
        result = predictor.predict_from_path('pill.jpg')
        # or from camera frame:
        result = predictor.predict_from_array(np_array_rgb)
    """

    def __init__(self, output_dir: str):
        with open(os.path.join(output_dir, 'model3_config.json')) as f:
            self.cfg = json.load(f)

        self.model = keras.models.load_model(
            os.path.join(output_dir, 'model3_best.keras'), compile=False
        )
        le = LabelEncoder()
        le.classes_ = np.load(os.path.join(output_dir, 'label_encoder.npy'),
                              allow_pickle=True)
        self.le          = le
        self.tok_m1      = AutoTokenizer.from_pretrained(self.cfg['m1_tokenizer'])
        self.tok_m2      = AutoTokenizer.from_pretrained(self.cfg['m2_tokenizer'])
        self.img_size    = tuple(self.cfg['img_size'])
        self.max_txt_len = self.cfg['max_text_len']
        print('✅ DrugPredictor loaded and ready.')

    def _prep_image(self, img: Image.Image) -> np.ndarray:
        return np.expand_dims(
            np.array(img.convert('RGB').resize(self.img_size), dtype=np.float32) / 255.0,
            axis=0
        )  # (1, 224, 224, 3)

    def _prep_text(self, text: str, tokenizer):
        enc = tokenizer(text, max_length=self.max_txt_len,
                        padding='max_length', truncation=True,
                        return_tensors='np')
        return (enc['input_ids'].astype(np.float32),
                enc['attention_mask'].astype(np.float32))

    def predict(self, image: Image.Image,
                text: str = 'a pharmaceutical drug',
                top_k: int = 3) -> dict:
        """
        Args:
            image : PIL Image (RGB)
            text  : optional description / context (can be empty for camera use)
            top_k : return top-k predictions
        Returns:
            dict with 'top_class', 'confidence', 'top_k_predictions'
        """
        img_arr            = self._prep_image(image)
        m1_ids,  m1_mask   = self._prep_text(text, self.tok_m1)
        m2_ids,  m2_mask   = self._prep_text(text, self.tok_m2)

        probs = self.model.predict(
            [img_arr, m1_ids, m1_mask, m2_ids, m2_mask], verbose=0
        )[0]  # (num_classes,)

        top_idx  = np.argsort(probs)[::-1][:top_k]
        top_pred = [
            {'class': self.le.classes_[i], 'confidence': float(probs[i])}
            for i in top_idx
        ]

        return {
            'top_class'        : top_pred[0]['class'],
            'confidence'       : top_pred[0]['confidence'],
            'top_k_predictions': top_pred
        }

    def predict_from_path(self, path: str, text: str = 'a pharmaceutical drug') -> dict:
        return self.predict(Image.open(path), text)

    def predict_from_array(self, rgb_array: np.ndarray,
                           text: str = 'a pharmaceutical drug') -> dict:
        """For live camera frames (numpy H×W×3 uint8 or float32)."""
        if rgb_array.dtype != np.uint8:
            rgb_array = (rgb_array * 255).astype(np.uint8)
        return self.predict(Image.fromarray(rgb_array), text)


# ── Quick smoke test ──────────────────────────────────────────────────────
predictor = DrugPredictor(OUTPUT_DIR)

# Test with first image from test set
result = predictor.predict_from_path(test_img[0], text=test_txt[0])
true_label = le.classes_[test_y[0]]

print(f'\nSample Image  : {test_img[0]}')
print(f'True Label    : {true_label}')
print(f'Predicted     : {result["top_class"]}  ({result["confidence"]*100:.1f}% confidence)')
print(f'Top-{len(result["top_k_predictions"])} :')
for p in result['top_k_predictions']:
    print(f'  {p["class"]:<30} {p["confidence"]*100:.2f}%')

---
## 📸 Cell 19 — Quick Camera Simulation Test

In [ ]:
# Simulates receiving a frame from a camera / uploaded image
# Replace test_img[0] with your actual image path or camera frame

from PIL import Image as PILImage

sample_image_path = test_img[0]   # ← swap with your image
pil_img = PILImage.open(sample_image_path)

# Show the image inline
plt.figure(figsize=(4, 4))
plt.imshow(pil_img)
plt.axis('off')
plt.title('Input Image', fontsize=12)
plt.show()

# Run prediction (text is optional — for camera-only use, leave default)
result = predictor.predict(pil_img, text='a pharmaceutical drug')

print(f'\n🔍 Prediction Result')
print(f'   Drug Name   : {result["top_class"]}')
print(f'   Confidence  : {result["confidence"]*100:.2f}%')
print(f'\n   Top Predictions:')
for rank, p in enumerate(result['top_k_predictions'], 1):
    bar = '█' * int(p['confidence'] * 30)
    print(f'   #{rank}  {p["class"]:<28} {bar} {p["confidence"]*100:.1f}%')

---
## 📊 Cell 20 — Model 3 vs Model 1 vs Model 2 Comparison
> Paste your Model 1 and Model 2 test accuracies below.

In [ ]:
# ── Fill in from your classification_report.txt files ─────────────────────
MODEL1_TEST_ACC = 0.82    # ← replace with your actual value
MODEL2_TEST_ACC = 0.85    # ← replace with your actual value
MODEL3_TEST_ACC = test_acc

models = ['Model 1\n(EfficientNet+DistilBERT)', 
          'Model 2\n(MobileNet+BERT)',
          'Model 3\n(Attention Fusion)']
accs   = [MODEL1_TEST_ACC, MODEL2_TEST_ACC, MODEL3_TEST_ACC]
colors = ['steelblue', 'coral', 'mediumseagreen']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, [a * 100 for a in accs], color=colors,
              edgecolor='black', linewidth=0.7, width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{acc*100:.1f}%', ha='center', va='bottom',
            fontsize=13, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison — Test Set Accuracy', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150)
plt.show()

print(f'\nModel 1 Test Acc : {MODEL1_TEST_ACC*100:.1f}%')
print(f'Model 2 Test Acc : {MODEL2_TEST_ACC*100:.1f}%')
print(f'Model 3 Test Acc : {MODEL3_TEST_ACC*100:.1f}%')
print(f'\nGain over Model 1: {(MODEL3_TEST_ACC - MODEL1_TEST_ACC)*100:+.1f}%')
print(f'Gain over Model 2: {(MODEL3_TEST_ACC - MODEL2_TEST_ACC)*100:+.1f}%')

---
## 📁 Cell 21 — Summary of Output Files

In [ ]:
print('='*55)
print('   MODEL 3 — DEPLOYMENT ARTIFACTS SUMMARY')
print('='*55)
artifacts = [
    ('model3_best.keras',                 'Trained Model 3 (load for inference)'),
    ('label_encoder.npy',                 'Class name ↔ index mapping'),
    ('model3_config.json',                'Config: tokenizer paths, img size, etc.'),
    ('model3_classification_report.txt',  'Per-class precision/recall/F1'),
    ('model3_training_log.csv',           'Epoch-by-epoch training log'),
    ('model3_training_history.png',       'Acc/Loss training curves'),
    ('model3_confusion_matrix.png',       'Confusion matrix heatmap'),
    ('model3_gate_analysis.png',          'α/β gate weight visualisation'),
    ('model_comparison.png',              'Model 1 vs 2 vs 3 bar chart'),
]
for fname, desc in artifacts:
    full = os.path.join(OUTPUT_DIR, fname)
    flag = '✅' if os.path.exists(full) else '⏳'
    print(f'  {flag}  {fname:<45} {desc}')

print('\n✨ Hand the DrugPredictor class + model3_output/ folder to your deployment team.')